In [4]:
import os
import numpy as np
import rasterio # type: ignore
from collections import defaultdict

bands_dir = "Data/bands"
files = sorted(os.listdir(bands_dir))

# Collect available dates — need B08, B11, and SCL
date_files = defaultdict(dict)
for f in files:
    if not f.endswith(".tif"):
        continue
    name = f.replace(".tif", "")
    date_str, band = name.rsplit("_", 1)
    date_files[date_str][band] = os.path.join(bands_dir, f)

# Only keep dates that have all three bands
all_dates = sorted(d for d in date_files if {"B08", "B11", "SCL"} <= date_files[d].keys())
print(f"Found {len(all_dates)} dates with B08 + B11 + SCL")

# Already filtered at download time (70% clear), just load everything
band_data = {}
dates = []

print("Loading …")
for i, date_str in enumerate(all_dates):
    with rasterio.open(date_files[date_str]["B08"]) as src:
        b08 = src.read(1)
    with rasterio.open(date_files[date_str]["B11"]) as src:
        b11 = src.read(1)
    with rasterio.open(date_files[date_str]["SCL"]) as src:
        scl = src.read(1)

    band_data[date_str] = {"B08": b08, "B11": b11, "SCL": scl}
    dates.append(date_str)

    if (i + 1) % 50 == 0 or (i + 1) == len(all_dates):
        print(f"  loaded {i + 1}/{len(all_dates)}")

print(f"Done. {len(dates)} dates ready")
if dates:
    print(f"Range: {dates[0]} to {dates[-1]}")

Found 51 dates with B08 + B11 + SCL
Loading …
  loaded 50/51
  loaded 51/51
Done. 51 dates ready
Range: 2018-06-13 to 2025-08-12


In [ ]:
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 150  # 50 MB limit
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from rasterio.warp import transform as warp_transform # type: ignore

BAD_SCL = {0, 1, 2, 3, 8, 9, 10, 11}

# Precompute all NDMI frames
ndmi_frames = []
for date in dates:
    entry = band_data[date]
    b08 = entry["B08"].astype(np.float32)
    b11 = entry["B11"].astype(np.float32)
    scl = entry["SCL"]
    bad_mask = np.isin(scl, list(BAD_SCL)) | (b08 == 0)
    denom = b08 + b11
    ndmi = np.where((denom > 0) & ~bad_mask, (b08 - b11) / denom, np.nan)
    ndmi_frames.append(ndmi)

# Load TI Koatinemo boundary from local file
with open("Data/ti_koatinemo.geojson") as f:
    geojson = json.load(f)

ref_path = date_files[dates[-1]]["B08"]
with rasterio.open(ref_path) as src:
    raster_transform = src.transform

rings_px = []
for feat in geojson["features"]:
    geom = feat["geometry"]
    polygons = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    for polygon in polygons:
        for ring in polygon:
            lons = [pt[0] for pt in ring]
            lats = [pt[1] for pt in ring]
            xs, ys = warp_transform("EPSG:4674", "EPSG:32722", lons, lats)
            inv = ~raster_transform
            cols, rows = zip(*(inv * (x, y) for x, y in zip(xs, ys)))
            rings_px.append((cols, rows))

print(f"Got {len(geojson['features'])} feature(s), {len(rings_px)} ring(s)")

# Render NDMI timelapse with boundary overlay
print(f"Rendering timelapse with {len(ndmi_frames)} frames …")
h, w = ndmi_frames[0].shape

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(ndmi_frames[0], cmap="RdYlGn", vmin=-0.5, vmax=0.7, interpolation="nearest")
ax.axis("off")
title = ax.set_title(f"NDMI — {dates[0]}", fontsize=14)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for cols, rows in rings_px:
    ax.plot(cols, rows, color="black", linewidth=1.5, solid_capstyle="round")
ax.set_xlim(0, w)
ax.set_ylim(h, 0)
fig.tight_layout()
plt.close(fig)

def update(idx):
    im.set_data(ndmi_frames[idx])
    title.set_text(f"NDMI — {dates[idx]}  ({idx + 1}/{len(dates)})")
    return [im, title]

anim = FuncAnimation(fig, update, frames=len(dates), interval=800, blit=True)
HTML(anim.to_jshtml())

In [6]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.ndimage import binary_dilation

# Dilate cloud mask to remove cloud-edge artifacts (haze, thin cloud, shadow fringes)
ndmi_cls_frames = []
for frame in ndmi_frames:
    cloud_mask = np.isnan(frame)
    dilated = binary_dilation(cloud_mask, iterations=3)
    clean = frame.copy()
    clean[dilated] = np.nan
    ndmi_cls_frames.append(clean)

# 2 discrete classes: red (deforested), green (healthy)
cmap_cls = ListedColormap(["red", "limegreen"])
cmap_cls.set_bad("white")  # NaN → white (cloud/shadow)
norm_cls = BoundaryNorm([-1, 0, 1], cmap_cls.N)

print(f"Rendering classified NDMI timelapse with {len(ndmi_cls_frames)} frames …")

fig2, ax2 = plt.subplots(figsize=(10, 8))
im2 = ax2.imshow(ndmi_cls_frames[0], cmap=cmap_cls, norm=norm_cls, interpolation="nearest")
ax2.axis("off")
title2 = ax2.set_title(f"Classified NDMI — {dates[0]}", fontsize=14)

# Legend-style colorbar with class labels
cb = fig2.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, ticks=[-0.5, 0.5])
cb.ax.set_yticklabels(["Deforested", "Healthy"])

# Overlay TI Koatinemo boundary (clipped to AOI)
for cols, rows in rings_px:
    ax2.plot(cols, rows, color="black", linewidth=1.5, solid_capstyle="round")
ax2.set_xlim(0, w)
ax2.set_ylim(h, 0)

fig2.tight_layout()
plt.close(fig2)

def update_cls(idx):
    im2.set_data(ndmi_cls_frames[idx])
    title2.set_text(f"Classified NDMI — {dates[idx]}  ({idx + 1}/{len(dates)})")
    return [im2, title2]

anim2 = FuncAnimation(fig2, update_cls, frames=len(dates), interval=800, blit=True)
HTML(anim2.to_jshtml())

Rendering classified NDMI timelapse with 51 frames …
